# Variational Equilibrium Analysis of AI Integration in Pharmaceutical Education Using OULAD — Q1 Selective Upgrade

This upgraded notebook preserves the original OULAD-based variational educational intelligence pipeline and adds a selective Q1-oriented experimental layer. The added layer explicitly compares conventional AI and VEAI, models temporal equilibrium dynamics, constructs variational energy landscapes, evaluates perturbation robustness, and generates adaptive educational governance outputs.

The notebook assumes the dataset is stored at:

`/content/drive/MyDrive/Datasets/OULAD/`


In [ ]:
# Cell 1 — Environment setup and Google Drive mounting

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")
np.random.seed(42)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path('/content/drive/MyDrive/Datasets/OULAD') if IN_COLAB else Path.cwd() / 'OULAD'
OUT_DIR = Path('/content/drive/MyDrive/Outputs/VEAI_Pharmaceutical_Education') if IN_COLAB else Path.cwd() / 'VEAI_Pharmaceutical_Education'
FIG_DIR = OUT_DIR / 'figures'
TABLE_DIR = OUT_DIR / 'tables'
LOG_DIR = OUT_DIR / 'logs'

for d in [OUT_DIR, FIG_DIR, TABLE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

summary_path = OUT_DIR / 'outputs_summary.txt'

def log(msg):
    print(msg)
    with open(summary_path, 'a', encoding='utf-8') as f:
        f.write(str(msg) + '\n')

with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('Variational Equilibrium Analysis of AI Integration in Pharmaceutical Education\n')
    f.write('='*80 + '\n')

log(f'Base dataset path: {BASE_DIR}')
log(f'Output path: {OUT_DIR}')

In [ ]:
# Cell 2 — Load OULAD files

required_files = {
    'assessments': 'assessments.csv',
    'courses': 'courses.csv',
    'studentAssessment': 'studentAssessment.csv',
    'studentInfo': 'studentInfo.csv',
    'studentRegistration': 'studentRegistration.csv',
    'vle': 'vle.csv'
}

data = {}
for key, fname in required_files.items():
    path = BASE_DIR / fname
    if not path.exists():
        raise FileNotFoundError(f'Missing file: {path}')
    data[key] = pd.read_csv(path)
    log(f'{fname}: {data[key].shape}')

# Load all split studentVle files and concatenate
vle_parts = sorted(BASE_DIR.glob('studentVle_*.csv'))
if len(vle_parts) == 0:
    single_vle = BASE_DIR / 'studentVle.csv'
    if single_vle.exists():
        student_vle = pd.read_csv(single_vle)
    else:
        raise FileNotFoundError('No studentVle_*.csv or studentVle.csv found.')
else:
    parts = []
    for p in vle_parts:
        df = pd.read_csv(p)
        parts.append(df)
        log(f'{p.name}: {df.shape}')
    student_vle = pd.concat(parts, ignore_index=True)

data['studentVle'] = student_vle
log(f'Combined studentVle: {student_vle.shape}')

In [ ]:
# Cell 3 — Initial data overview

for name, df in data.items():
    log('\n' + '-'*80)
    log(f'{name}')
    log(df.head().to_string())
    log(f'Columns: {list(df.columns)}')
    log(f'Missing values:\n{df.isna().sum().to_string()}')

In [ ]:
# Cell 4 — Build assessment-performance features

assessments = data['assessments'].copy()
student_assess = data['studentAssessment'].copy()
student_info = data['studentInfo'].copy()
registration = data['studentRegistration'].copy()

assess_merged = student_assess.merge(
    assessments,
    on='id_assessment',
    how='left',
    suffixes=('_student', '_assessment')
)

# Weighted score contribution
assess_merged['weighted_score'] = assess_merged['score'] * assess_merged['weight'] / 100.0

perf_features = assess_merged.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    mean_score=('score', 'mean'),
    max_score=('score', 'max'),
    min_score=('score', 'min'),
    std_score=('score', 'std'),
    n_assessments_submitted=('id_assessment', 'count'),
    total_weighted_score=('weighted_score', 'sum'),
    mean_submission_date=('date_submitted', 'mean'),
    late_submissions=('is_banked', 'sum')
).reset_index()

perf_features['std_score'] = perf_features['std_score'].fillna(0)
log(f'Performance features: {perf_features.shape}')
perf_features.to_csv(TABLE_DIR / 'table_performance_features.csv', index=False)

In [ ]:
# Cell 5 — Build engagement features from VLE interactions

student_vle = data['studentVle'].copy()
vle = data['vle'].copy()

vle_merged = student_vle.merge(vle, on='id_site', how='left')

eng_features = vle_merged.groupby(['code_module', 'code_presentation', 'id_student']).agg(
    total_clicks=('sum_click', 'sum'),
    mean_clicks=('sum_click', 'mean'),
    active_days=('date', 'nunique'),
    first_activity_day=('date', 'min'),
    last_activity_day=('date', 'max'),
    n_activity_records=('id_site', 'count'),
    n_unique_sites=('id_site', 'nunique'),
    n_activity_types=('activity_type', 'nunique')
).reset_index()

eng_features['engagement_span'] = eng_features['last_activity_day'] - eng_features['first_activity_day']
eng_features['clicks_per_active_day'] = eng_features['total_clicks'] / eng_features['active_days'].replace(0, np.nan)
eng_features['clicks_per_active_day'] = eng_features['clicks_per_active_day'].fillna(0)

log(f'Engagement features: {eng_features.shape}')
eng_features.to_csv(TABLE_DIR / 'table_engagement_features.csv', index=False)

In [ ]:
# Cell 6 — Registration and persistence features

reg_features = registration.copy()

reg_features['withdrawn'] = reg_features['date_unregistration'].notna().astype(int)
reg_features['registration_duration'] = reg_features['date_unregistration'].fillna(270) - reg_features['date_registration']
reg_features['registration_duration'] = reg_features['registration_duration'].clip(lower=0)

reg_features = reg_features[['code_module', 'code_presentation', 'id_student',
                             'date_registration', 'date_unregistration',
                             'withdrawn', 'registration_duration']]

log(f'Registration features: {reg_features.shape}')
reg_features.to_csv(TABLE_DIR / 'table_registration_features.csv', index=False)

In [ ]:
# Cell 7 — Assemble learner-level analytical dataset

df = student_info.merge(perf_features, on=['code_module', 'code_presentation', 'id_student'], how='left')
df = df.merge(eng_features, on=['code_module', 'code_presentation', 'id_student'], how='left')
df = df.merge(reg_features, on=['code_module', 'code_presentation', 'id_student'], how='left')

numeric_cols = [
    'mean_score', 'max_score', 'min_score', 'std_score', 'n_assessments_submitted',
    'total_weighted_score', 'mean_submission_date', 'late_submissions',
    'total_clicks', 'mean_clicks', 'active_days', 'first_activity_day',
    'last_activity_day', 'n_activity_records', 'n_unique_sites',
    'n_activity_types', 'engagement_span', 'clicks_per_active_day',
    'withdrawn', 'registration_duration'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df['success'] = df['final_result'].isin(['Pass', 'Distinction']).astype(int)
df['failure_or_withdrawal'] = df['final_result'].isin(['Fail', 'Withdrawn']).astype(int)

log(f'Final analytical dataset: {df.shape}')
log(df['final_result'].value_counts().to_string())
df.to_csv(TABLE_DIR / 'table_oulad_analytical_dataset.csv', index=False)

In [ ]:
# Cell 8 — Construct UBIF/variational-equilibrium variables

def minmax(s):
    s = pd.Series(s).astype(float)
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

df['P_performance'] = minmax(df['total_weighted_score'])
df['G_engagement'] = minmax(np.log1p(df['total_clicks']))
df['R_persistence'] = minmax(df['registration_duration'])
df['S_stability'] = 1 - minmax(df['std_score'])
df['D_disengagement'] = 1 - minmax(df['active_days'])
df['O_overload_proxy'] = minmax(df['clicks_per_active_day']) * (1 - df['P_performance'])

# Educational variational equilibrium score
alpha, beta, gamma, rho, delta, eta = 0.25, 0.20, 0.20, 0.15, 0.10, 0.10

df['E_equilibrium'] = (
    alpha * df['P_performance'] +
    beta  * df['G_engagement'] +
    gamma * df['R_persistence'] +
    rho   * df['S_stability'] -
    delta * df['D_disengagement'] -
    eta   * df['O_overload_proxy']
)

df['E_equilibrium_norm'] = minmax(df['E_equilibrium'])

df[['id_student', 'code_module', 'code_presentation', 'final_result',
    'P_performance', 'G_engagement', 'R_persistence', 'S_stability',
    'D_disengagement', 'O_overload_proxy', 'E_equilibrium_norm']].to_csv(
    TABLE_DIR / 'table_ubif_equilibrium_variables.csv', index=False
)

log('UBIF/variational-equilibrium variables constructed.')
log(df[['P_performance','G_engagement','R_persistence','S_stability','D_disengagement','O_overload_proxy','E_equilibrium_norm']].describe().to_string())

In [ ]:
# Cell 9 — Visualize equilibrium distributions

plt.figure(figsize=(8, 5))
plt.hist(df['E_equilibrium_norm'], bins=40)
plt.xlabel('Normalized Educational Equilibrium')
plt.ylabel('Number of students')
plt.title('Distribution of UBIF Educational Equilibrium Scores')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_equilibrium_distribution.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
means = df.groupby('final_result')['E_equilibrium_norm'].mean().sort_values()
means.plot(kind='bar')
plt.xlabel('Final result')
plt.ylabel('Mean normalized equilibrium')
plt.title('Educational Equilibrium by Final Outcome')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_equilibrium_by_outcome.png', dpi=300)
plt.show()

In [ ]:
# Cell 10 — Learner-state clustering

cluster_cols = [
    'P_performance', 'G_engagement', 'R_persistence',
    'S_stability', 'D_disengagement', 'O_overload_proxy'
]

X = df[cluster_cols].replace([np.inf, -np.inf], 0).fillna(0)
X_scaled = StandardScaler().fit_transform(X)

scores = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

best_k = max(scores, key=scores.get)
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
df['learner_state'] = kmeans.fit_predict(X_scaled)

state_summary = df.groupby('learner_state')[cluster_cols + ['E_equilibrium_norm', 'success', 'withdrawn']].mean()
state_summary.to_csv(TABLE_DIR / 'table_learner_state_summary.csv')

log(f'Best number of learner states: {best_k}')
log(state_summary.to_string())

plt.figure(figsize=(8, 5))
state_summary['E_equilibrium_norm'].plot(kind='bar')
plt.xlabel('Learner state')
plt.ylabel('Mean normalized equilibrium')
plt.title('Mean Educational Equilibrium by Learner State')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_learner_state_equilibrium.png', dpi=300)
plt.show()

In [ ]:
# Cell 11 — Predictive validity check: does equilibrium help explain outcomes?

features_base = ['P_performance', 'G_engagement', 'R_persistence', 'S_stability',
                 'D_disengagement', 'O_overload_proxy', 'E_equilibrium_norm']

model_df = df[features_base + ['success']].replace([np.inf, -np.inf], 0).fillna(0)

X = model_df[features_base]
y = model_df['success']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

bal_acc = balanced_accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred)

log(f'Balanced accuracy: {bal_acc:.4f}')
log(f'F1 score: {f1:.4f}')
log('\nClassification report:\n' + classification_report(y_test, pred))

imp = pd.Series(clf.feature_importances_, index=features_base).sort_values(ascending=True)
imp.to_csv(TABLE_DIR / 'table_feature_importance.csv')

plt.figure(figsize=(8, 5))
imp.plot(kind='barh')
plt.xlabel('Feature importance')
plt.title('Predictive Importance of UBIF Educational Variables')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_feature_importance.png', dpi=300)
plt.show()

In [ ]:
# Cell 12 — AI integration simulation: no AI, balanced AI, excessive AI

sim = df.copy()

def simulate_ai_condition(data, condition):
    out = data.copy()
    if condition == 'No AI':
        out['AI_support'] = 0.00
        out['AI_dependence'] = 0.00
        out['governance'] = 0.20
    elif condition == 'General-purpose AI':
        out['AI_support'] = 0.12
        out['AI_dependence'] = 0.18
        out['governance'] = 0.35
    elif condition == 'Balanced supervised AI':
        out['AI_support'] = 0.22
        out['AI_dependence'] = 0.10
        out['governance'] = 0.75
    elif condition == 'Excessive AI dependence':
        out['AI_support'] = 0.18
        out['AI_dependence'] = 0.35
        out['governance'] = 0.25
    else:
        raise ValueError(condition)

    # Conceptual perturbation equations
    out['P_sim'] = np.clip(out['P_performance'] + out['AI_support']*(1-out['P_performance']) - 0.20*out['AI_dependence'], 0, 1)
    out['G_sim'] = np.clip(out['G_engagement'] + 0.15*out['AI_support'] - 0.10*out['AI_dependence'], 0, 1)
    out['R_sim'] = np.clip(out['R_persistence'] + 0.10*out['AI_support'] + 0.10*out['governance'] - 0.15*out['AI_dependence'], 0, 1)
    out['D_sim'] = np.clip(out['D_disengagement'] - 0.10*out['AI_support'] - 0.10*out['governance'] + 0.20*out['AI_dependence'], 0, 1)
    out['O_sim'] = np.clip(out['O_overload_proxy'] + 0.25*out['AI_dependence'] - 0.15*out['governance'], 0, 1)

    out['E_sim'] = (
        alpha*out['P_sim'] +
        beta*out['G_sim'] +
        gamma*out['R_sim'] +
        rho*out['S_stability'] -
        delta*out['D_sim'] -
        eta*out['O_sim']
    )
    out['condition'] = condition
    return out

conditions = ['No AI', 'General-purpose AI', 'Balanced supervised AI', 'Excessive AI dependence']
sim_results = pd.concat([simulate_ai_condition(df, c) for c in conditions], ignore_index=True)
sim_results['E_sim_norm'] = sim_results.groupby('condition')['E_sim'].transform(lambda x: (x - x.min())/(x.max()-x.min()) if x.max()!=x.min() else 0)

scenario_summary = sim_results.groupby('condition').agg(
    mean_equilibrium=('E_sim', 'mean'),
    std_equilibrium=('E_sim', 'std'),
    mean_performance=('P_sim', 'mean'),
    mean_engagement=('G_sim', 'mean'),
    mean_persistence=('R_sim', 'mean'),
    mean_disengagement=('D_sim', 'mean'),
    mean_overload=('O_sim', 'mean')
).reset_index()

scenario_summary.to_csv(TABLE_DIR / 'table_ai_scenario_summary.csv', index=False)
log('\nAI scenario summary:')
log(scenario_summary.to_string(index=False))

In [ ]:
# Cell 13 — Plot AI scenario comparison

order = ['No AI', 'General-purpose AI', 'Balanced supervised AI', 'Excessive AI dependence']
scenario_summary['condition'] = pd.Categorical(scenario_summary['condition'], categories=order, ordered=True)
scenario_summary = scenario_summary.sort_values('condition')

plt.figure(figsize=(9, 5))
plt.bar(scenario_summary['condition'], scenario_summary['mean_equilibrium'])
plt.ylabel('Mean simulated educational equilibrium')
plt.title('AI Integration Scenarios Under Variational Equilibrium')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_ai_scenario_equilibrium.png', dpi=300)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(scenario_summary['condition'], scenario_summary['mean_performance'], marker='o', label='Performance')
plt.plot(scenario_summary['condition'], scenario_summary['mean_engagement'], marker='o', label='Engagement')
plt.plot(scenario_summary['condition'], scenario_summary['mean_persistence'], marker='o', label='Persistence')
plt.plot(scenario_summary['condition'], scenario_summary['mean_overload'], marker='o', label='Overload/Dependence')
plt.ylabel('Mean normalized value')
plt.title('Scenario-Level UBIF Components')
plt.xticks(rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_ai_scenario_components.png', dpi=300)
plt.show()

In [ ]:
# Cell 14 — Variational phase diagram: support versus dependence

support_grid = np.linspace(0, 0.35, 40)
dependence_grid = np.linspace(0, 0.40, 40)

Z = np.zeros((len(dependence_grid), len(support_grid)))

base = df.sample(min(5000, len(df)), random_state=42).copy()

for i, dep in enumerate(dependence_grid):
    for j, sup in enumerate(support_grid):
        gov = 0.60
        P_sim = np.clip(base['P_performance'] + sup*(1-base['P_performance']) - 0.20*dep, 0, 1)
        G_sim = np.clip(base['G_engagement'] + 0.15*sup - 0.10*dep, 0, 1)
        R_sim = np.clip(base['R_persistence'] + 0.10*sup + 0.10*gov - 0.15*dep, 0, 1)
        D_sim = np.clip(base['D_disengagement'] - 0.10*sup - 0.10*gov + 0.20*dep, 0, 1)
        O_sim = np.clip(base['O_overload_proxy'] + 0.25*dep - 0.15*gov, 0, 1)

        E = alpha*P_sim + beta*G_sim + gamma*R_sim + rho*base['S_stability'] - delta*D_sim - eta*O_sim
        Z[i, j] = E.mean()

plt.figure(figsize=(8, 6))
plt.imshow(
    Z,
    origin='lower',
    aspect='auto',
    extent=[support_grid.min(), support_grid.max(), dependence_grid.min(), dependence_grid.max()]
)
plt.colorbar(label='Mean educational equilibrium')
plt.xlabel('AI support intensity')
plt.ylabel('AI dependence intensity')
plt.title('Variational Phase Diagram: AI Support vs Dependence')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_variational_phase_diagram.png', dpi=300)
plt.show()

phase_df = pd.DataFrame(Z, index=np.round(dependence_grid, 3), columns=np.round(support_grid, 3))
phase_df.to_csv(TABLE_DIR / 'table_variational_phase_diagram.csv')

In [ ]:
# Cell 15 — Time-window engagement trajectories

# Aggregate VLE clicks into learning-time windows
vle_time = data['studentVle'].copy()
vle_time = vle_time[vle_time['date'].notna()].copy()
vle_time['time_window'] = pd.cut(
    vle_time['date'],
    bins=[-999, 0, 30, 60, 90, 120, 180, 999],
    labels=['pre-course', '0-30', '31-60', '61-90', '91-120', '121-180', '180+']
)

traj = vle_time.groupby(['code_module', 'code_presentation', 'id_student', 'time_window'])['sum_click'].sum().reset_index()
traj = traj.merge(df[['code_module','code_presentation','id_student','final_result']], 
                  on=['code_module','code_presentation','id_student'], how='left')

traj_summary = traj.groupby(['time_window', 'final_result'])['sum_click'].mean().reset_index()
traj_summary.to_csv(TABLE_DIR / 'table_engagement_trajectories.csv', index=False)

plt.figure(figsize=(9, 5))
for outcome in traj_summary['final_result'].dropna().unique():
    tmp = traj_summary[traj_summary['final_result'] == outcome]
    plt.plot(tmp['time_window'].astype(str), tmp['sum_click'], marker='o', label=outcome)

plt.xlabel('Learning time window')
plt.ylabel('Mean clicks')
plt.title('Engagement Trajectories by Final Outcome')
plt.xticks(rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_engagement_trajectories.png', dpi=300)
plt.show()

## Selective Q1-Oriented Upgrade Layer

The following cells add a publication-oriented experimental layer without rebuilding the original pipeline. They create explicit conventional-AI versus variational-equilibrium AI comparisons, temporal equilibrium dynamics, variational energy landscapes, perturbation robustness tests, and educational governance outputs.

In [ ]:
# Cell 16 — Q1 upgrade setup: helper metrics, calibration, and plotting utilities

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, roc_auc_score, brier_score_loss,
    log_loss, precision_score, recall_score, accuracy_score
)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

UPGRADE_FIG_DIR = FIG_DIR
UPGRADE_TABLE_DIR = TABLE_DIR

q1_summary_lines = []

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    records = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi if i < n_bins - 1 else y_prob <= hi)
        if mask.sum() == 0:
            records.append({'bin': i, 'lo': lo, 'hi': hi, 'n': 0, 'confidence': np.nan, 'accuracy': np.nan, 'gap': np.nan})
            continue
        conf = y_prob[mask].mean()
        acc = y_true[mask].mean()
        gap = abs(acc - conf)
        ece += mask.mean() * gap
        records.append({'bin': i, 'lo': lo, 'hi': hi, 'n': int(mask.sum()), 'confidence': conf, 'accuracy': acc, 'gap': gap})
    return float(ece), pd.DataFrame(records)

def evaluate_probabilistic_model(name, y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    ece, _ = expected_calibration_error(y_true, y_prob, n_bins=10)
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'brier_score': brier_score_loss(y_true, y_prob),
        'log_loss': log_loss(y_true, np.clip(y_prob, 1e-6, 1-1e-6)),
        'ece_10_bins': ece,
    }

q1_summary_lines.append('Q1 upgrade layer initialized: metrics, calibration, and helper functions ready.')


In [ ]:
# Cell 17 — Conventional AI vs VEAI comparative predictive layer

# Conventional AI uses standard learning-analytics indicators without explicit equilibrium.
conventional_features = [
    'mean_score', 'total_weighted_score', 'n_assessments_submitted', 'std_score',
    'total_clicks', 'mean_clicks', 'active_days', 'n_activity_records',
    'n_unique_sites', 'n_activity_types', 'clicks_per_active_day',
    'registration_duration', 'withdrawn', 'num_of_prev_attempts', 'studied_credits'
]
conventional_features = [c for c in conventional_features if c in df.columns]

# VEAI uses equilibrium-aware variables and the derived variational-equilibrium score.
veai_features = [
    'P_performance', 'G_engagement', 'R_persistence', 'S_stability',
    'D_disengagement', 'O_overload_proxy', 'E_equilibrium_norm'
]

comparison_df = df[conventional_features + veai_features + ['success']].replace([np.inf, -np.inf], 0).fillna(0)
y = comparison_df['success'].astype(int)

idx_train, idx_test = train_test_split(
    comparison_df.index, test_size=0.25, random_state=42, stratify=y
)

Xc_train = comparison_df.loc[idx_train, conventional_features]
Xc_test  = comparison_df.loc[idx_test, conventional_features]
Xv_train = comparison_df.loc[idx_train, veai_features]
Xv_test  = comparison_df.loc[idx_test, veai_features]
y_train = y.loc[idx_train]
y_test  = y.loc[idx_test]

scaler_c = StandardScaler()
Xc_train_s = scaler_c.fit_transform(Xc_train)
Xc_test_s = scaler_c.transform(Xc_test)

scaler_v = StandardScaler()
Xv_train_s = scaler_v.fit_transform(Xv_train)
Xv_test_s = scaler_v.transform(Xv_test)

models = {
    'Conventional Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Conventional Random Forest': RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced', n_jobs=-1),
    'Conventional Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'VEAI Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'VEAI Random Forest': RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced', n_jobs=-1),
    'VEAI Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

comparison_records = []
model_probabilities = {}
trained_models = {}

for name, model in models.items():
    if name.startswith('Conventional'):
        Xtr = Xc_train_s if 'Logistic' in name else Xc_train
        Xte = Xc_test_s if 'Logistic' in name else Xc_test
    else:
        Xtr = Xv_train_s if 'Logistic' in name else Xv_train
        Xte = Xv_test_s if 'Logistic' in name else Xv_test
    model.fit(Xtr, y_train)
    prob = model.predict_proba(Xte)[:, 1]
    comparison_records.append(evaluate_probabilistic_model(name, y_test, prob))
    model_probabilities[name] = prob
    trained_models[name] = model

comparison_metrics = pd.DataFrame(comparison_records).sort_values('balanced_accuracy', ascending=False)
comparison_metrics.to_csv(UPGRADE_TABLE_DIR / 'table_q1_conventional_vs_veai_metrics.csv', index=False)
log('\nQ1 comparative predictive metrics:')
log(comparison_metrics.round(4).to_string(index=False))

# Figure: comparative performance metrics
plot_metrics = ['balanced_accuracy', 'f1', 'roc_auc', 'ece_10_bins']
metric_plot = comparison_metrics.set_index('model')[plot_metrics]
plt.figure(figsize=(11, 6))
metric_plot.plot(kind='bar', ax=plt.gca())
plt.ylabel('Score')
plt.title('Conventional AI versus VEAI: Predictive, Reliability, and Calibration Metrics')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_conventional_vs_veai_metrics.png', dpi=300)
plt.show()

# Figure: confidence and equilibrium comparison
best_conventional_name = comparison_metrics[comparison_metrics['model'].str.startswith('Conventional')].iloc[0]['model']
best_veai_name = comparison_metrics[comparison_metrics['model'].str.startswith('VEAI')].iloc[0]['model']

plt.figure(figsize=(9, 5))
plt.hist(model_probabilities[best_conventional_name], bins=30, alpha=0.65, label=best_conventional_name)
plt.hist(model_probabilities[best_veai_name], bins=30, alpha=0.65, label=best_veai_name)
plt.xlabel('Predicted success probability')
plt.ylabel('Number of learners')
plt.title('Conventional Confidence versus VEAI Equilibrium-Aware Confidence')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_confidence_distribution_comparison.png', dpi=300)
plt.show()

q1_summary_lines.append('Added conventional AI versus VEAI predictive, calibration, and confidence comparison.')


In [ ]:
# Cell 18 — Calibration and risk-coverage analysis

calibration_records = []
plt.figure(figsize=(8, 6))
for name in [best_conventional_name, best_veai_name]:
    prob = model_probabilities[name]
    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10, strategy='uniform')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)
    ece, cal_df = expected_calibration_error(y_test, prob, n_bins=10)
    cal_df['model'] = name
    calibration_records.append(cal_df)

plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed success rate')
plt.title('Calibration Reliability: Conventional AI versus VEAI')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_calibration_reliability_comparison.png', dpi=300)
plt.show()

calibration_table = pd.concat(calibration_records, ignore_index=True)
calibration_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_calibration_bins.csv', index=False)

# Selective prediction / risk-coverage analysis
risk_records = []
coverages = np.linspace(0.50, 1.00, 11)
for name in [best_conventional_name, best_veai_name]:
    prob = model_probabilities[name]
    confidence = np.maximum(prob, 1 - prob)
    pred = (prob >= 0.5).astype(int)
    for cov in coverages:
        threshold = np.quantile(confidence, 1 - cov)
        mask = confidence >= threshold
        risk = 1 - accuracy_score(y_test[mask], pred[mask])
        risk_records.append({
            'model': name,
            'coverage': cov,
            'risk': risk,
            'n_retained': int(mask.sum()),
            'mean_confidence': confidence[mask].mean()
        })

risk_table = pd.DataFrame(risk_records)
risk_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_risk_coverage.csv', index=False)

plt.figure(figsize=(8, 5))
for name in [best_conventional_name, best_veai_name]:
    tmp = risk_table[risk_table['model'] == name]
    plt.plot(tmp['coverage'], tmp['risk'], marker='o', label=name)
plt.xlabel('Coverage retained')
plt.ylabel('Prediction risk')
plt.title('Risk-Coverage Tradeoff for Selective Educational Decision Support')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_risk_coverage_tradeoff.png', dpi=300)
plt.show()

q1_summary_lines.append('Added calibration reliability curves and risk-coverage analysis.')


In [ ]:
# Cell 19 — Temporal variational-equilibrium dynamics

# Weekly/periodic learner interaction windows are converted into dynamic equilibrium trajectories.
vle_time = data['studentVle'].copy()
vle_time = vle_time[vle_time['date'].notna()].copy()
vle_time['time_window'] = pd.cut(
    vle_time['date'],
    bins=[-999, 0, 30, 60, 90, 120, 180, 999],
    labels=['pre-course', '0-30', '31-60', '61-90', '91-120', '121-180', '180+']
)

window_clicks = vle_time.groupby(['code_module', 'code_presentation', 'id_student', 'time_window']).agg(
    window_clicks=('sum_click', 'sum'),
    window_active_days=('date', 'nunique'),
    window_records=('id_site', 'count')
).reset_index()

static_cols = ['code_module', 'code_presentation', 'id_student', 'final_result', 'success',
               'P_performance', 'R_persistence', 'S_stability', 'O_overload_proxy', 'learner_state']
temporal = window_clicks.merge(df[static_cols], on=['code_module', 'code_presentation', 'id_student'], how='left')

temporal['G_window'] = minmax(np.log1p(temporal['window_clicks']))
temporal['D_window'] = 1 - minmax(temporal['window_active_days'])
temporal['O_window'] = np.clip(temporal['O_overload_proxy'] + minmax(temporal['window_records']) * 0.10, 0, 1)
temporal['E_window'] = (
    alpha * temporal['P_performance'] +
    beta  * temporal['G_window'] +
    gamma * temporal['R_persistence'] +
    rho   * temporal['S_stability'] -
    delta * temporal['D_window'] -
    eta   * temporal['O_window']
)
temporal['E_window_norm'] = minmax(temporal['E_window'])
temporal.to_csv(UPGRADE_TABLE_DIR / 'table_q1_temporal_equilibrium_learner_windows.csv', index=False)

temporal_summary = temporal.groupby(['time_window', 'final_result']).agg(
    mean_equilibrium=('E_window_norm', 'mean'),
    std_equilibrium=('E_window_norm', 'std'),
    mean_window_clicks=('window_clicks', 'mean'),
    mean_active_days=('window_active_days', 'mean'),
    n_learners=('id_student', 'nunique')
).reset_index()
temporal_summary.to_csv(UPGRADE_TABLE_DIR / 'table_q1_temporal_equilibrium_summary.csv', index=False)

plt.figure(figsize=(10, 5))
for outcome in temporal_summary['final_result'].dropna().unique():
    tmp = temporal_summary[temporal_summary['final_result'] == outcome]
    plt.plot(tmp['time_window'].astype(str), tmp['mean_equilibrium'], marker='o', label=outcome)
plt.xlabel('Learning time window')
plt.ylabel('Mean temporal equilibrium')
plt.title('Temporal Variational-Equilibrium Trajectories by Final Outcome')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_temporal_equilibrium_trajectories.png', dpi=300)
plt.show()

# Heatmap-like state evolution across windows
state_window = temporal.groupby(['time_window', 'learner_state'])['E_window_norm'].mean().unstack()
state_window.to_csv(UPGRADE_TABLE_DIR / 'table_q1_state_window_equilibrium.csv')
plt.figure(figsize=(8, 5))
plt.imshow(state_window.T, aspect='auto', origin='lower')
plt.colorbar(label='Mean temporal equilibrium')
plt.yticks(range(len(state_window.columns)), [f'State {s}' for s in state_window.columns])
plt.xticks(range(len(state_window.index)), [str(x) for x in state_window.index], rotation=35, ha='right')
plt.xlabel('Learning time window')
plt.ylabel('Learner state')
plt.title('Learner-State Stability Evolution Across Learning Windows')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_state_stability_heatmap.png', dpi=300)
plt.show()

q1_summary_lines.append('Added temporal equilibrium trajectories and learner-state stability heatmap.')


In [ ]:
# Cell 20 — Learner-state transition and early-warning indicators

# Early warning is defined from low equilibrium, high disengagement, and weak engagement in each time window.
temporal['early_warning_score'] = np.clip(
    0.45 * (1 - temporal['E_window_norm']) +
    0.35 * temporal['D_window'] +
    0.20 * (1 - temporal['G_window']), 0, 1
)
temporal['warning_level'] = pd.cut(
    temporal['early_warning_score'],
    bins=[-0.01, 0.33, 0.66, 1.01],
    labels=['Low', 'Moderate', 'High']
)

warning_summary = temporal.groupby(['time_window', 'warning_level']).agg(
    n_records=('id_student', 'count'),
    withdrawal_rate=('final_result', lambda x: np.mean(x == 'Withdrawn')),
    failure_rate=('final_result', lambda x: np.mean(x == 'Fail')),
    success_rate=('success', 'mean'),
    mean_early_warning=('early_warning_score', 'mean')
).reset_index()
warning_summary.to_csv(UPGRADE_TABLE_DIR / 'table_q1_early_warning_summary.csv', index=False)

plt.figure(figsize=(10, 5))
for level in ['Low', 'Moderate', 'High']:
    tmp = warning_summary[warning_summary['warning_level'] == level]
    plt.plot(tmp['time_window'].astype(str), tmp['withdrawal_rate'], marker='o', label=f'{level} warning')
plt.xlabel('Learning time window')
plt.ylabel('Withdrawal rate')
plt.title('Withdrawal Dynamics Under Temporal Early-Warning Levels')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_withdrawal_early_warning_dynamics.png', dpi=300)
plt.show()

# Transition matrix from static learner states to warning levels.
transition_table = pd.crosstab(temporal['learner_state'], temporal['warning_level'], normalize='index')
transition_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_state_warning_transition_probabilities.csv')

plt.figure(figsize=(7, 5))
plt.imshow(transition_table.values, aspect='auto', origin='lower')
plt.colorbar(label='Transition probability')
plt.xticks(range(len(transition_table.columns)), transition_table.columns.astype(str))
plt.yticks(range(len(transition_table.index)), [f'State {s}' for s in transition_table.index])
plt.xlabel('Temporal warning level')
plt.ylabel('Learner state')
plt.title('Learner-State to Warning-Level Transition Probabilities')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_state_warning_transition_map.png', dpi=300)
plt.show()

q1_summary_lines.append('Added early-warning dynamics and state-to-warning transition probabilities.')


In [ ]:
# Cell 21 — Variational energy landscape and equilibrium surface

# The energy landscape formalizes the tradeoff between supportive AI and dependence/overload.
support_grid = np.linspace(0, 0.40, 60)
dependence_grid = np.linspace(0, 0.45, 60)
landscape = np.zeros((len(dependence_grid), len(support_grid)))
instability = np.zeros_like(landscape)

base_land = df.sample(min(6000, len(df)), random_state=42).copy()

for i, dep in enumerate(dependence_grid):
    for j, sup in enumerate(support_grid):
        gov = np.clip(0.45 + 0.80 * sup - 0.70 * dep, 0, 1)
        P_sim = np.clip(base_land['P_performance'] + sup*(1-base_land['P_performance']) - 0.22*dep, 0, 1)
        G_sim = np.clip(base_land['G_engagement'] + 0.18*sup - 0.12*dep, 0, 1)
        R_sim = np.clip(base_land['R_persistence'] + 0.10*sup + 0.12*gov - 0.17*dep, 0, 1)
        D_sim = np.clip(base_land['D_disengagement'] - 0.12*sup - 0.12*gov + 0.24*dep, 0, 1)
        O_sim = np.clip(base_land['O_overload_proxy'] + 0.32*dep - 0.18*gov, 0, 1)
        E = alpha*P_sim + beta*G_sim + gamma*R_sim + rho*base_land['S_stability'] - delta*D_sim - eta*O_sim
        landscape[i, j] = E.mean()
        instability[i, j] = np.mean((D_sim > 0.75) | (O_sim > 0.35) | (E < np.quantile(base_land['E_equilibrium'], 0.25)))

landscape_df = pd.DataFrame(landscape, index=np.round(dependence_grid, 4), columns=np.round(support_grid, 4))
landscape_df.to_csv(UPGRADE_TABLE_DIR / 'table_q1_variational_energy_landscape.csv')
instability_df = pd.DataFrame(instability, index=np.round(dependence_grid, 4), columns=np.round(support_grid, 4))
instability_df.to_csv(UPGRADE_TABLE_DIR / 'table_q1_instability_landscape.csv')

plt.figure(figsize=(8, 6))
cs = plt.contourf(support_grid, dependence_grid, landscape, levels=20)
plt.colorbar(cs, label='Mean variational equilibrium')
plt.xlabel('AI support intensity')
plt.ylabel('AI dependence intensity')
plt.title('Variational Energy Landscape of AI-Supported Learning')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_variational_energy_landscape.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
cs = plt.contourf(support_grid, dependence_grid, instability, levels=20)
plt.colorbar(cs, label='Instability probability')
plt.xlabel('AI support intensity')
plt.ylabel('AI dependence intensity')
plt.title('Educational Instability Surface Under Support-Dependence Dynamics')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_instability_surface.png', dpi=300)
plt.show()

# 3D surface, useful for graphical abstract or methodology figure.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
S_grid, D_grid = np.meshgrid(support_grid, dependence_grid)
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(S_grid, D_grid, landscape, linewidth=0, antialiased=True, alpha=0.9)
ax.set_xlabel('AI support')
ax.set_ylabel('AI dependence')
ax.set_zlabel('Equilibrium')
ax.set_title('3D Variational Equilibrium Surface')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_3d_equilibrium_surface.png', dpi=300)
plt.show()

q1_summary_lines.append('Added variational energy landscape, instability surface, and 3D equilibrium surface.')


In [ ]:
# Cell 22 — Robustness and perturbation analysis

rng = np.random.default_rng(42)
noise_levels = np.linspace(0, 0.40, 9)
robust_records = []

# Use the best conventional and VEAI models already trained.
for noise in noise_levels:
    # Conventional perturbation: noisy standard features.
    Xc_noisy = Xc_test.copy().astype(float)
    for col in Xc_noisy.columns:
        scale = Xc_noisy[col].std() if Xc_noisy[col].std() > 0 else 1.0
        Xc_noisy[col] = Xc_noisy[col] + rng.normal(0, noise * scale, size=len(Xc_noisy))
    if 'Logistic' in best_conventional_name:
        Xc_eval = scaler_c.transform(Xc_noisy)
    else:
        Xc_eval = Xc_noisy
    prob_c = trained_models[best_conventional_name].predict_proba(Xc_eval)[:, 1]
    rec_c = evaluate_probabilistic_model(best_conventional_name, y_test, prob_c)
    rec_c['noise_level'] = noise
    rec_c['family'] = 'Conventional AI'
    robust_records.append(rec_c)

    # VEAI perturbation: noise injected into equilibrium components, clipped to admissible ranges.
    Xv_noisy = Xv_test.copy().astype(float)
    for col in Xv_noisy.columns:
        Xv_noisy[col] = np.clip(Xv_noisy[col] + rng.normal(0, noise, size=len(Xv_noisy)), 0, 1)
    # Reconstruct equilibrium after perturbation to retain variational consistency.
    Xv_noisy['E_equilibrium_norm'] = minmax(
        alpha*Xv_noisy['P_performance'] + beta*Xv_noisy['G_engagement'] +
        gamma*Xv_noisy['R_persistence'] + rho*Xv_noisy['S_stability'] -
        delta*Xv_noisy['D_disengagement'] - eta*Xv_noisy['O_overload_proxy']
    )
    if 'Logistic' in best_veai_name:
        Xv_eval = scaler_v.transform(Xv_noisy[veai_features])
    else:
        Xv_eval = Xv_noisy[veai_features]
    prob_v = trained_models[best_veai_name].predict_proba(Xv_eval)[:, 1]
    rec_v = evaluate_probabilistic_model(best_veai_name, y_test, prob_v)
    rec_v['noise_level'] = noise
    rec_v['family'] = 'VEAI'
    robust_records.append(rec_v)

robustness_table = pd.DataFrame(robust_records)
robustness_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_noise_robustness_metrics.csv', index=False)

plt.figure(figsize=(8, 5))
for family in robustness_table['family'].unique():
    tmp = robustness_table[robustness_table['family'] == family]
    plt.plot(tmp['noise_level'], tmp['balanced_accuracy'], marker='o', label=family)
plt.xlabel('Perturbation intensity')
plt.ylabel('Balanced accuracy')
plt.title('Robustness of Conventional AI and VEAI Under Feature Perturbation')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_noise_robustness_balanced_accuracy.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
for family in robustness_table['family'].unique():
    tmp = robustness_table[robustness_table['family'] == family]
    plt.plot(tmp['noise_level'], tmp['ece_10_bins'], marker='o', label=family)
plt.xlabel('Perturbation intensity')
plt.ylabel('Expected calibration error')
plt.title('Calibration Degradation Under Perturbation')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_noise_robustness_calibration.png', dpi=300)
plt.show()

q1_summary_lines.append('Added perturbation robustness metrics and calibration-degradation curves.')


In [ ]:
# Cell 23 — Missingness and overload perturbation analysis

missing_levels = np.linspace(0, 0.50, 6)
missing_records = []

for miss in missing_levels:
    Xv_missing = Xv_test.copy().astype(float)
    mask = rng.random(Xv_missing[['G_engagement', 'P_performance']].shape) < miss
    gp = Xv_missing[['G_engagement', 'P_performance']].values
    gp[mask] = np.nan
    Xv_missing[['G_engagement', 'P_performance']] = gp
    Xv_missing = Xv_missing.fillna(Xv_train.median())
    Xv_missing['E_equilibrium_norm'] = minmax(
        alpha*Xv_missing['P_performance'] + beta*Xv_missing['G_engagement'] +
        gamma*Xv_missing['R_persistence'] + rho*Xv_missing['S_stability'] -
        delta*Xv_missing['D_disengagement'] - eta*Xv_missing['O_overload_proxy']
    )
    if 'Logistic' in best_veai_name:
        X_eval = scaler_v.transform(Xv_missing[veai_features])
    else:
        X_eval = Xv_missing[veai_features]
    prob = trained_models[best_veai_name].predict_proba(X_eval)[:, 1]
    rec = evaluate_probabilistic_model('VEAI under missingness', y_test, prob)
    rec['missing_rate'] = miss
    missing_records.append(rec)

missing_table = pd.DataFrame(missing_records)
missing_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_missingness_sensitivity.csv', index=False)

plt.figure(figsize=(8, 5))
plt.plot(missing_table['missing_rate'], missing_table['balanced_accuracy'], marker='o', label='Balanced accuracy')
plt.plot(missing_table['missing_rate'], missing_table['f1'], marker='o', label='F1')
plt.xlabel('Missing engagement/performance rate')
plt.ylabel('Metric value')
plt.title('VEAI Sensitivity to Missing Educational Evidence')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_missingness_sensitivity.png', dpi=300)
plt.show()

# Overload perturbation: simulate increasing dependence-induced overload and observe equilibrium degradation.
overload_levels = np.linspace(0, 0.50, 11)
overload_records = []
base_eval = Xv_test.copy()
for overload in overload_levels:
    pert = base_eval.copy()
    pert['O_overload_proxy'] = np.clip(pert['O_overload_proxy'] + overload, 0, 1)
    pert['D_disengagement'] = np.clip(pert['D_disengagement'] + 0.50 * overload, 0, 1)
    pert['G_engagement'] = np.clip(pert['G_engagement'] - 0.25 * overload, 0, 1)
    e_raw = alpha*pert['P_performance'] + beta*pert['G_engagement'] + gamma*pert['R_persistence'] + rho*pert['S_stability'] - delta*pert['D_disengagement'] - eta*pert['O_overload_proxy']
    pert['E_equilibrium_norm'] = minmax(e_raw)
    overload_records.append({
        'overload_intensity': overload,
        'mean_equilibrium': pert['E_equilibrium_norm'].mean(),
        'low_equilibrium_rate': (pert['E_equilibrium_norm'] < 0.33).mean(),
        'mean_disengagement': pert['D_disengagement'].mean(),
        'mean_engagement': pert['G_engagement'].mean()
    })

overload_table = pd.DataFrame(overload_records)
overload_table.to_csv(UPGRADE_TABLE_DIR / 'table_q1_overload_degradation.csv', index=False)

plt.figure(figsize=(8, 5))
plt.plot(overload_table['overload_intensity'], overload_table['mean_equilibrium'], marker='o', label='Mean equilibrium')
plt.plot(overload_table['overload_intensity'], overload_table['low_equilibrium_rate'], marker='o', label='Low-equilibrium rate')
plt.xlabel('AI-dependence / overload perturbation')
plt.ylabel('Value')
plt.title('Equilibrium Degradation Under Excessive AI Dependence')
plt.legend()
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_overload_equilibrium_degradation.png', dpi=300)
plt.show()

q1_summary_lines.append('Added missingness sensitivity and overload-degradation analysis.')


In [ ]:
# Cell 24 — Educational governance and intervention recommendation layer

policy_df = df.copy()
policy_df['risk_index'] = np.clip(
    0.40 * (1 - policy_df['E_equilibrium_norm']) +
    0.25 * policy_df['D_disengagement'] +
    0.20 * policy_df['O_overload_proxy'] +
    0.15 * (1 - policy_df['G_engagement']), 0, 1
)

conditions = [
    (policy_df['risk_index'] >= 0.70) & (policy_df['O_overload_proxy'] >= 0.15),
    (policy_df['risk_index'] >= 0.70),
    (policy_df['risk_index'] >= 0.50) & (policy_df['G_engagement'] < 0.45),
    (policy_df['risk_index'] >= 0.50),
    (policy_df['risk_index'] < 0.50) & (policy_df['E_equilibrium_norm'] >= 0.65),
]
choices = [
    'Reduce AI overload + intensive human tutoring',
    'Immediate intervention and advisor contact',
    'Engagement recovery plan',
    'Moderate adaptive AI support',
    'Autonomous enrichment and monitoring'
]
policy_df['recommended_policy'] = np.select(conditions, choices, default='Light monitoring')
policy_df['intervention_intensity'] = pd.cut(
    policy_df['risk_index'], bins=[-0.01, 0.33, 0.66, 1.01], labels=['Low', 'Moderate', 'High']
)

policy_summary = policy_df.groupby(['recommended_policy', 'intervention_intensity']).agg(
    n_learners=('id_student', 'count'),
    mean_equilibrium=('E_equilibrium_norm', 'mean'),
    mean_risk=('risk_index', 'mean'),
    withdrawal_rate=('final_result', lambda x: np.mean(x == 'Withdrawn')),
    success_rate=('success', 'mean'),
    mean_overload=('O_overload_proxy', 'mean')
).reset_index().sort_values(['mean_risk', 'n_learners'], ascending=[False, False])
policy_summary.to_csv(UPGRADE_TABLE_DIR / 'table_q1_intervention_policy_scenarios.csv', index=False)

plt.figure(figsize=(10, 5))
policy_counts = policy_df['recommended_policy'].value_counts().sort_values()
policy_counts.plot(kind='barh')
plt.xlabel('Number of learners')
plt.ylabel('Recommended educational governance policy')
plt.title('VEAI Educational Intervention Policy Distribution')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_intervention_policy_distribution.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
scatter = plt.scatter(policy_df['E_equilibrium_norm'], policy_df['risk_index'], s=8, alpha=0.35)
plt.xlabel('Variational equilibrium')
plt.ylabel('Governance risk index')
plt.title('Equilibrium-Risk Map for Adaptive Educational Governance')
plt.tight_layout()
plt.savefig(UPGRADE_FIG_DIR / 'fig_q1_equilibrium_risk_policy_map.png', dpi=300)
plt.show()

q1_summary_lines.append('Added educational governance policy map and intervention scenario table.')


In [ ]:
# Cell 25 — Q1 manuscript-ready upgrade summary

figures = sorted([p.name for p in FIG_DIR.glob('*.png')])
tables = sorted([p.name for p in TABLE_DIR.glob('*.csv')])

q1_manifest = pd.DataFrame({
    'artifact_type': ['figure'] * len(figures) + ['table'] * len(tables),
    'filename': figures + tables,
    'q1_upgrade': [('q1_' in f) for f in figures + tables]
})
q1_manifest.to_csv(TABLE_DIR / 'table_q1_output_manifest.csv', index=False)

log('\n' + '='*80)
log('Selective Q1 Upgrade Completed')
for line in q1_summary_lines:
    log('- ' + line)

log('\nGenerated Figures')
for f in figures:
    log(f'- {f}')

log('\nGenerated Tables')
for t in tables:
    log(f'- {t}')

log('\nSuggested upgraded manuscript results subsections:')
log('1. Public Dataset and Learning-Analytics Cohort Construction')
log('2. Variational Educational Equilibrium Variables')
log('3. Learner-State Discovery and Educational Stability')
log('4. Conventional AI versus VEAI: Predictive, Calibration, and Risk-Coverage Comparison')
log('5. Temporal Variational Dynamics and Early-Warning Behavior')
log('6. Variational Energy Landscape and Educational Instability Surface')
log('7. Robustness Under Perturbation, Missingness, and AI Overload')
log('8. Adaptive Educational Governance and Intervention Policy Mapping')

with open(summary_path, 'a', encoding='utf-8') as f:
    f.write('\n\n' + '='*80 + '\n')
    f.write('Selective Q1 Upgrade Completed\n')
    for line in q1_summary_lines:
        f.write('- ' + line + '\n')
    f.write('\nGenerated Figures\n')
    for fig in figures:
        f.write(f'- {fig}\n')
    f.write('\nGenerated Tables\n')
    for tab in tables:
        f.write(f'- {tab}\n')
    f.write('\nSuggested upgraded manuscript results subsections:\n')
    f.write('1. Public Dataset and Learning-Analytics Cohort Construction\n')
    f.write('2. Variational Educational Equilibrium Variables\n')
    f.write('3. Learner-State Discovery and Educational Stability\n')
    f.write('4. Conventional AI versus VEAI: Predictive, Calibration, and Risk-Coverage Comparison\n')
    f.write('5. Temporal Variational Dynamics and Early-Warning Behavior\n')
    f.write('6. Variational Energy Landscape and Educational Instability Surface\n')
    f.write('7. Robustness Under Perturbation, Missingness, and AI Overload\n')
    f.write('8. Adaptive Educational Governance and Intervention Policy Mapping\n')

log('\nNotebook completed successfully with selective Q1-oriented upgrade.')
